# Notebook 7 — Glued active `v0` sweep for several `Dtheta` values, `B = 50 R_inter`

## 0. Imports, paths, and run controls

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from front_runner import compile_c_code, run_front_simulations

project_folder = Path('.').resolve()
c_file = project_folder / 'abm_front_propagation_glued.c'
exe_file = project_folder / 'abm_front_propagation_glued_B50.exe'

sweep_param_file = project_folder / 'params_active_v0_Dtheta_sweep_glued_T1000.txt'
run_table_file = project_folder / 'params_active_v0_Dtheta_sweep_glued_T1000.csv'
param_base = sweep_param_file.stem

data_folder = project_folder / 'data' / param_base
data_folder.mkdir(parents=True, exist_ok=True)

figure_folder = project_folder / 'figures' / param_base
figure_folder.mkdir(parents=True, exist_ok=True)

compile_first = False
run_simulations = False

selected_run_ids = None

skip_existing_outputs = True

max_parallel = 25

print('Project folder:', project_folder)
print('C file:', c_file)
print('Parameter base:', param_base)
print('Data folder:', data_folder)
print('Figure folder:', figure_folder)

if not c_file.exists():
    raise FileNotFoundError(f'Cannot find {c_file}. Put abm_front_propagation_glued.c in this folder.')

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Base active-matter parameters

In [ ]:
base = {
    'run_id': 0,
    'seed': 69069,

    'p0': 0.85,
    'q0': 0.15,
    'Ns': 50,
    'R_inter': 0.05,
    'rho0': 750.0,

    'save_per_step': 10000,
    'front_per_step': 200,
    'rho_profile_every_front': 10,

    'isolation_buffer_factor': 50.0,

    'v0': 0.010,
    'Dr': 0.00006325,
    'Dtheta': 0.002,

    'dt': 0.001,
    'T': 1000.0,

    'Lx': 80.0,
    'Ly': 2.0,
    'x_init_min': 39.75,
    'x_init_max': 40.25,

    'warmup_T': 200.0,
    'rho_sat': -1.0,

    'nbins_x': 1600,
    'threshold_frac1': 0.2,
    'threshold_frac2': 0.5,
    'threshold_frac3': 0.8,
}

r_edge = base['p0'] - base['q0']
isolation_buffer = base['isolation_buffer_factor'] * base['R_inter']
initial_area = (base['x_init_max'] - base['x_init_min']) * base['Ly']
initial_N0 = int(base['rho0'] * initial_area + 0.5)

print(f'r = p0 - q0 = {r_edge:g}')
print(f'R_inter = {base["R_inter"]:g}')
print(f'isolation_buffer_factor = {base["isolation_buffer_factor"]:g}')
print(f'isolation buffer B = {isolation_buffer:g} = {base["isolation_buffer_factor"]:g} * R_inter')
print(f'dt = {base["dt"]:g}')
print(f'Dr = {base["Dr"]:g}')
print(f'Lx = {base["Lx"]:g}, Ly = {base["Ly"]:g}, dx_bin = {base["Lx"]/base["nbins_x"]:g}')
print(f'x_init = [{base["x_init_min"]:g}, {base["x_init_max"]:g}], N0 ≈ {initial_N0}')
print(f'Brownian FKPP reference speed = {2*np.sqrt(base["Dr"]*r_edge):.8g}')

## 2. Sweep choices

In [ ]:
Dtheta_values = np.array([0.002, 0.05, 0.5, 2.0, 5.0], dtype=float)

v0_values = np.array([0.0, 0.006, 0.008, 0.010, 0.012], dtype=float)

seed_values = np.array([69069, 15015015, 123458], dtype=int)


print('Dtheta values:', Dtheta_values)
print('v0 values:', v0_values)
print('Seeds:', seed_values)
print('Total runs:', len(Dtheta_values) * len(v0_values) * len(seed_values))

sweep_info = []
for Dtheta in Dtheta_values:
    for v0 in v0_values:
        D_eff = base['Dr'] + v0**2 / (2.0 * Dtheta)
        sweep_info.append({
            'Dtheta': Dtheta,
            'v0': v0,
            'persistence_time_1_over_Dtheta': 1.0 / Dtheta,
            'persistence_length_v0_over_Dtheta': v0 / Dtheta,
            'D_eff_active': D_eff,
            'v_FKPP_brownian': 2.0 * np.sqrt(base['Dr'] * r_edge),
            'v_FKPP_active_eff': 2.0 * np.sqrt(D_eff * r_edge),
        })

pd.DataFrame(sweep_info)

## 3. Timestep and small-step checks

In [ ]:
def active_v0_Dtheta_dt_limits(base, v0_values, Dtheta_values, fraction=0.1):
    """Mirror the C-code small-step checks.

    The C code stops if:
        v0*dt > 0.1*R_inter
        sqrt(2*Dr*dt) > 0.1*R_inter
        sqrt(2*Dtheta*dt) > 0.1
        p0*dt > 0.01 or q0*dt > 0.01
    """
    R_inter = float(base['R_inter'])
    Dr = float(base['Dr'])
    p0 = float(base['p0'])
    q0 = float(base['q0'])
    v0_max = float(np.max(v0_values))
    Dtheta_max = float(np.max(Dtheta_values))

    limits = {}
    limits['active: v0_max*dt < 0.1 R_inter'] = np.inf if v0_max <= 0 else fraction * R_inter / v0_max
    limits['translation: sqrt(2 Dr dt) < 0.1 R_inter'] = np.inf if Dr <= 0 else (fraction * R_inter)**2 / (2.0 * Dr)
    limits['angle: sqrt(2 Dtheta_max dt) < 0.1'] = np.inf if Dtheta_max <= 0 else fraction**2 / (2.0 * Dtheta_max)
    limits['birth/death: p0*dt,q0*dt < 0.01'] = 0.01 / max(p0, q0) if max(p0, q0) > 0 else np.inf
    return limits

chosen_dt = float(base['dt'])
dt_limits = active_v0_Dtheta_dt_limits(base, v0_values, Dtheta_values, fraction=0.1)
dt_max = min(dt_limits.values())
limiting_name = min(dt_limits, key=dt_limits.get)

print('dt limits from the C-code 10% conditions:')
for name, value in dt_limits.items():
    print(f'{name:52s}: {value:g}')
print()
print(f'dt_max = {dt_max:g}')
print(f'limiting condition: {limiting_name}')
print(f'chosen dt = {chosen_dt:g}')

v0_max = float(np.max(v0_values))
Dtheta_max = float(np.max(Dtheta_values))

check_table = pd.DataFrame([
    {
        'condition': 'v0_max * dt',
        'value': v0_max * chosen_dt,
        'limit': 0.1 * base['R_inter'],
        'ratio_value_over_limit': (v0_max * chosen_dt) / (0.1 * base['R_inter']),
    },
    {
        'condition': 'sqrt(2 Dr dt)',
        'value': np.sqrt(2.0 * base['Dr'] * chosen_dt),
        'limit': 0.1 * base['R_inter'],
        'ratio_value_over_limit': np.sqrt(2.0 * base['Dr'] * chosen_dt) / (0.1 * base['R_inter']),
    },
    {
        'condition': 'sqrt(2 Dtheta_max dt)',
        'value': np.sqrt(2.0 * Dtheta_max * chosen_dt),
        'limit': 0.1,
        'ratio_value_over_limit': np.sqrt(2.0 * Dtheta_max * chosen_dt) / 0.1,
    },
    {
        'condition': 'p0 * dt',
        'value': base['p0'] * chosen_dt,
        'limit': 0.01,
        'ratio_value_over_limit': (base['p0'] * chosen_dt) / 0.01,
    },
    {
        'condition': 'q0 * dt',
        'value': base['q0'] * chosen_dt,
        'limit': 0.01,
        'ratio_value_over_limit': (base['q0'] * chosen_dt) / 0.01,
    },
])

display(check_table)

geometry_check = pd.DataFrame([
    {
        'condition': 'R_inter < Ly/2',
        'value': base['R_inter'],
        'limit': base['Ly'] / 2.0,
        'passes': base['R_inter'] < base['Ly'] / 2.0,
    },
    {
        'condition': 'R_inter < initial_width/2',
        'value': base['R_inter'],
        'limit': (base['x_init_max'] - base['x_init_min']) / 2.0,
        'passes': base['R_inter'] < (base['x_init_max'] - base['x_init_min']) / 2.0,
    },
    {
        'condition': 'x_init inside [0,Lx]',
        'value': base['x_init_min'],
        'limit': base['Lx'],
        'passes': 0.0 <= base['x_init_min'] < base['x_init_max'] <= base['Lx'],
    },
])
display(geometry_check)

if chosen_dt > dt_max:
    raise ValueError('Chosen dt violates at least one C-code timestep condition.')
elif not geometry_check['passes'].all():
    raise ValueError('Chosen geometry violates one of the C-code geometry checks.')
else:
    print('Chosen dt and geometry pass the C-code checks.')

## 4. Write the parameter file

In [ ]:
run_base = dict(base)
run_base['dt'] = chosen_dt

param_order = [
    'run_id', 'seed',
    'p0', 'q0', 'Ns', 'R_inter', 'rho0',
    'save_per_step', 'front_per_step', 'rho_profile_every_front',
    'isolation_buffer_factor',
    'v0', 'Dr', 'Dtheta', 'dt', 'T',
    'Lx', 'Ly', 'x_init_min', 'x_init_max', 'warmup_T',
    'rho_sat', 'nbins_x', 'threshold_frac1', 'threshold_frac2', 'threshold_frac3',
]

def active_theory_metadata(v0, Dtheta, Dr, r_edge):
    D_eff = Dr + v0**2 / (2.0 * Dtheta)
    return {
        'D_eff': D_eff,
        'persistence_time': 1.0 / Dtheta,
        'persistence_length': v0 / Dtheta,
        'v_fkpp_brownian': 2.0 * np.sqrt(Dr * r_edge),
        'v_fkpp_active_eff': 2.0 * np.sqrt(D_eff * r_edge),
    }

runs = []
run_id = 0
for Dtheta in Dtheta_values:
    for v0 in v0_values:
        for seed in seed_values:
            run = dict(run_base)
            run['run_id'] = run_id
            run['Dtheta'] = float(Dtheta)
            run['v0'] = float(v0)
            run['seed'] = int(seed)
            runs.append(run)
            run_id += 1

with open(sweep_param_file, 'w') as f:
    f.write('# Active matter v0 x Dtheta sweep using abm_front_propagation_glued.c.\n')
    f.write('# Fixed choices: R_inter = 0.05, Lx = 80, Ly = 2.0, Dr = 0.00006325, dt = 0.001.\n')
    f.write('# isolation_buffer_factor = 50.0, so isolation buffer B = 50 * R_inter.\n')
    f.write('# Runs vary Dtheta, v0 and seed. Includes v0 = 0 for every Dtheta.\n')
    f.write(f'# dt_max from C-code 10 percent conditions = {dt_max:.16g}\n')
    f.write('# C-code conditions checked: active step, translational diffusion, angular diffusion, birth/death.\n\n')
    for run in runs:
        f.write('[run]\n')
        for key in param_order:
            f.write(f'{key} = {run[key]}\n')
        f.write('\n')

run_table_rows = []
for r in runs:
    meta = active_theory_metadata(
        v0=float(r['v0']),
        Dtheta=float(r['Dtheta']),
        Dr=float(r['Dr']),
        r_edge=float(r['p0'] - r['q0']),
    )
    run_table_rows.append({
        'run_id': r['run_id'],
        'Dtheta': r['Dtheta'],
        'v0': r['v0'],
        'Dr': r['Dr'],
        'p0': r['p0'],
        'q0': r['q0'],
        'r_edge': r['p0'] - r['q0'],
        'seed': r['seed'],
        'dt': r['dt'],
        'R_inter': r['R_inter'],
        'Lx': r['Lx'],
        'Ly': r['Ly'],
        'x_init_min': r['x_init_min'],
        'x_init_max': r['x_init_max'],
        'front_per_step': r['front_per_step'],
        'save_per_step': r['save_per_step'],
        'rho_profile_every_front': r['rho_profile_every_front'],
        'isolation_buffer_factor': r['isolation_buffer_factor'],
        'isolation_buffer': r['isolation_buffer_factor'] * r['R_inter'],
        **meta,
    })

run_table = pd.DataFrame(run_table_rows)
run_table.to_csv(run_table_file, index=False)

print(f'Wrote {len(runs)} runs to {sweep_param_file}')
print(f'Wrote run table to {run_table_file}')
run_table

## 5. Compile

In [ ]:
if compile_first:
    compile_c_code(c_file, exe_file, flags=['-O3', '-std=c99', '-Wall', '-Wextra'])
else:
    print('Skipping compilation.')

## 6. Run simulations

In [ ]:
if not run_simulations:
    print('run_simulations = False')
    print('Set run_simulations = True in the first cell when you are ready to launch the sweep.')
else:
    run_results = run_front_simulations(
        exe_file=exe_file,
        param_file=sweep_param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing_outputs,
        required_output='front',
    )
    run_results

# Analysis

## 7. Read front files and fit velocities

In [ ]:
from front_analysis import (
    fit_front_sweep,
    pooled_speed_summary_by_method,
    side_speed_long_table,
    style_axis,
)

main_method = 'th_2'
methods_to_check = ['th_1', 'th_2', 'th_3']
methods_all = ['tip', 'quantile', 'th_1', 'th_2', 'th_3', 'mass']

fit_t_min = None
fit_t_max = None
main_fit_window = (0.70, 0.95)
fit_windows = [(0.60, 0.90), (0.70, 0.90), (0.70, 0.95), (0.80, 0.95)]
fit_start_fraction, fit_end_fraction = main_fit_window

run_table = pd.read_csv(run_table_file)

fit_df, missing = fit_front_sweep(
    run_table,
    data_folder=data_folder,
    param_base=param_base,
    methods=methods_all,
    fit_t_min=fit_t_min,
    fit_t_max=fit_t_max,
    fit_start_fraction=fit_start_fraction,
    fit_end_fraction=fit_end_fraction,
)

if missing:
    print('Missing front files for run_id:', missing)

print(f'Fitted {len(fit_df)} runs out of {len(run_table)} planned runs.')
fit_df.head()

## 8. Pooled speed average over runs and sides for each `(Dtheta, v0)`

In [ ]:
def run_level_speed_summary_2d(
    fit_df,
    group_cols,
    method,
    metadata_cols=None,
):
    """
    First average the left and right fronts within each run.
    Then compute the mean, standard deviation, and SEM over
    independent runs.
    """
    if metadata_cols is None:
        metadata_cols = []

    grouped = list(group_cols)

    keep_columns = []
    for column in ["run_id"] + grouped + list(metadata_cols):
        if column in fit_df.columns and column not in keep_columns:
            keep_columns.append(column)

    speed_long = side_speed_long_table(
        fit_df,
        method=method,
        keep_columns=keep_columns,
    )

    if len(speed_long) == 0:
        return pd.DataFrame()

    speed_long = speed_long.replace(
        [np.inf, -np.inf],
        np.nan,
    )
    speed_long = speed_long.dropna(subset=["speed"])

    # First average left and right within each realization
    run_cols = ["run_id"] + grouped

    run_level = (
        speed_long
        .groupby(run_cols, as_index=False)
        .agg(
            run_speed=("speed", "mean"),
            n_front_sides=("speed", "count"),
        )
    )

    # Then calculate variability across independent realizations
    summary = (
        run_level
        .groupby(grouped, as_index=False)
        .agg(
            n_runs=("run_speed", "count"),
            mean_speed=("run_speed", "mean"),
            speed_std=("run_speed", "std"),
        )
    )

    summary["speed_sem"] = (
        summary["speed_std"]
        / np.sqrt(summary["n_runs"])
    )

    # Keep left/right information for the asymmetry analysis
    side_means = (
        speed_long
        .pivot_table(
            index=grouped,
            columns="side",
            values="speed",
            aggfunc="mean",
        )
        .reset_index()
        .rename(columns={
            "left": "left_mean",
            "right": "right_mean",
        })
    )

    side_stds = (
        speed_long
        .pivot_table(
            index=grouped,
            columns="side",
            values="speed",
            aggfunc="std",
        )
        .reset_index()
        .rename(columns={
            "left": "left_std",
            "right": "right_std",
        })
    )

    summary = summary.merge(
        side_means,
        on=grouped,
        how="left",
    )
    summary = summary.merge(
        side_stds,
        on=grouped,
        how="left",
    )

    meta_cols = [
        column
        for column in metadata_cols
        if column in fit_df.columns
        and column not in grouped
    ]

    if meta_cols:
        metadata = (
            fit_df
            .groupby(grouped, as_index=False)
            .agg({
                column: "first"
                for column in meta_cols
            })
        )

        summary = summary.merge(
            metadata,
            on=grouped,
            how="left",
        )

    summary["method"] = method
    summary["asymmetry"] = (
        summary["left_mean"]
        -
        summary["right_mean"]
    )

    summary["relative_asymmetry"] = (
        summary["asymmetry"]
        /
        summary["mean_speed"]
    )

    return summary.sort_values(grouped).reset_index(drop=True)

summary = run_level_speed_summary_2d(
    fit_df,
    group_cols=["Dtheta", "v0"],
    method=main_method,
    metadata_cols=[
        "Dr",
        "p0",
        "q0",
        "r_edge",
        "R_inter",
        "dt",
        "D_eff",
        "persistence_time",
        "persistence_length",
        "v_fkpp_brownian",
        "v_fkpp_active_eff",
        "isolation_buffer_factor",
        "isolation_buffer",
        "Lx",
        "Ly",
    ],
)

summary['speed_over_active_eff'] = summary['mean_speed'] / summary['v_fkpp_active_eff']
summary['v2_measured'] = summary['mean_speed'] ** 2
summary['v2_active_eff'] = summary['v_fkpp_active_eff'] ** 2
summary['speed_over_v0'] = np.where(summary['v0'] > 0, summary['mean_speed'] / summary['v0'], np.nan)
summary

## 9. Main normal-scale plot: front speed versus activity for several `Dtheta`

In [ ]:
error_column = 'speed_std'  # use 'speed_sem' if you prefer error bars on the mean
fit_include_v0_zero = False

fig, ax = plt.subplots(figsize=(12.5, 5.2))

linear_fit_rows = []

for Dtheta in Dtheta_values:
    tmp = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('v0').copy()
    if len(tmp) == 0:
        continue

    eb = ax.errorbar(
        tmp['v0'],
        tmp['mean_speed'],
        yerr=tmp[error_column],
        marker='o',
        linestyle='',
        capsize=3,
        label=fr'$D_\theta={Dtheta:g}$'
    )

    color = eb[0].get_color()
    valid = np.isfinite(tmp['v0']) & np.isfinite(tmp['mean_speed'])
    if not fit_include_v0_zero:
        valid &= tmp['v0'] > 0

    if valid.sum() >= 2:
        x_fit = tmp.loc[valid, 'v0'].to_numpy()
        y_fit = tmp.loc[valid, 'mean_speed'].to_numpy()
        slope, intercept = np.polyfit(x_fit, y_fit, 1)
        y_pred = slope * x_fit + intercept
        residuals = y_fit - y_pred
        rmse = np.sqrt(np.mean(residuals**2))

        v0_line = np.linspace(tmp.loc[valid, 'v0'].min(), tmp.loc[valid, 'v0'].max(), 200)
        ax.plot(v0_line, slope * v0_line + intercept,
                linestyle='--', linewidth=1.6, color=color)

        linear_fit_rows.append({
            'Dtheta': Dtheta,
            'slope': slope,
            'intercept': intercept,
            'rmse': rmse,
            'n_points': int(valid.sum()),
            'fit_include_v0_zero': fit_include_v0_zero,
            'method': main_method,
            'fit_start_fraction': fit_start_fraction,
            'fit_end_fraction': fit_end_fraction,
        })

brownian_ref = 2.0 * np.sqrt(base['Dr'] * r_edge)
ax.axhline(brownian_ref, linestyle=':', linewidth=1.4, label='Brownian FKPP reference')

ax.set_xscale('linear')
ax.set_yscale('linear')
ax.set_xlabel(r'$v_0$')
ax.set_ylabel(r'$\langle v_{out}\rangle_{runs,sides}$')
ax.set_title(fr'Glued active front, $B=50R_{{inter}}$, $R_{{inter}}=0.05$, $L_x=80$, $L_y=2$, method={main_method}')
ax.grid(False)

ax.legend(
    fontsize=11,
    loc='center left',
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    title=r'$D_\theta$'
)

fig.tight_layout(rect=[0.0, 0.0, 0.78, 1.0])

save_path = figure_folder / 'glued_B50_active_v0_Dtheta_velocity_vs_v0_normal_scale.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

linear_fit_df = pd.DataFrame(linear_fit_rows)
linear_fit_df

## 9b. Individual plots with linear fit, one per `Dtheta`

In [ ]:
def safe_float_label(x):
    return f'{x:g}'.replace('-', 'm').replace('.', 'p')


for Dtheta in Dtheta_values:
    tmp = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('v0').copy()
    if len(tmp) == 0:
        continue

    fig, ax = plt.subplots(figsize=(10.5, 4.8))

    ax.plot(tmp['v0'], tmp['left_mean'], marker='^', linestyle='', label='left mean')
    ax.plot(tmp['v0'], tmp['right_mean'], marker='v', linestyle='', label='right mean')
    ax.errorbar(
        tmp['v0'],
        tmp['mean_speed'],
        yerr=tmp[error_column],
        marker='o',
        linestyle='',
        capsize=3,
        label=fr'$\langle v_{{out}}\rangle_{{runs,sides}}$, {main_method}'
    )

    valid = np.isfinite(tmp['v0']) & np.isfinite(tmp['mean_speed'])
    if not fit_include_v0_zero:
        valid &= tmp['v0'] > 0

    if valid.sum() >= 2:
        x_fit = tmp.loc[valid, 'v0'].to_numpy()
        y_fit = tmp.loc[valid, 'mean_speed'].to_numpy()
        slope, intercept = np.polyfit(x_fit, y_fit, 1)
        y_pred = slope * x_fit + intercept
        rmse = np.sqrt(np.mean((y_fit - y_pred)**2))
        v0_line = np.linspace(tmp.loc[valid, 'v0'].min(), tmp.loc[valid, 'v0'].max(), 200)
        ax.plot(v0_line, slope * v0_line + intercept,
                linestyle='--', linewidth=2.0,
                label=fr'linear fit: $v={slope:.3g}v_0+{intercept:.3g}$, RMSE={rmse:.2g}')

    brownian_ref = 2.0 * np.sqrt(base['Dr'] * r_edge)
    ax.axhline(brownian_ref, linestyle=':', linewidth=1.2, label='Brownian FKPP reference')

    ax.set_xscale('linear')
    ax.set_yscale('linear')
    ax.set_xlabel(r'$v_0$')
    ax.set_ylabel(r'$\langle v_{out}\rangle_{runs,sides}$')
    ax.set_title(fr'$D_\theta={Dtheta:g}$, glued $B=50R_{{inter}}$, $R_{{inter}}=0.05$, $L_x=80$, $L_y=2$')
    ax.grid(False)

    ax.legend(
        fontsize=10,
        loc='center left',
        bbox_to_anchor=(1.02, 0.5),
        frameon=False
    )

    fig.tight_layout(rect=[0.0, 0.0, 0.74, 1.0])
    save_path = figure_folder / f'glued_B50_v0_sweep_linear_fit_Dtheta_{safe_float_label(Dtheta)}.png'
    fig.savefig(save_path, dpi=170, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 9c. Check the `v0 = 0` case across `Dtheta`

In [ ]:
zero_summary = summary[np.isclose(summary['v0'], 0.0)].sort_values('Dtheta').copy()

fig, ax = plt.subplots(figsize=(8.8, 4.8))

if len(zero_summary) > 0:
    ax.errorbar(
        zero_summary['Dtheta'],
        zero_summary['mean_speed'],
        yerr=zero_summary[error_column],
        marker='o',
        linestyle='-',
        capsize=3,
        label=r'measured $v_0=0$'
    )

brownian_ref = 2.0 * np.sqrt(base['Dr'] * r_edge)
ax.axhline(brownian_ref, linestyle='--', linewidth=1.4,
           label=fr'Brownian FKPP reference = {brownian_ref:.4g}')

ax.set_xscale('linear')
ax.set_yscale('linear')
ax.set_xlabel(r'$D_\theta$')
ax.set_ylabel(r'$\langle v_{out}\rangle_{runs,sides}$')
ax.set_title(fr'$v_0=0$ diagnostic, glued $B=50R_{{inter}}$, normal scale')
ax.grid(False)
ax.legend(fontsize=11)
fig.tight_layout()

save_path = figure_folder / 'glued_B50_v0_zero_speed_vs_Dtheta_normal_scale.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

zero_summary

## 10. Left-right asymmetry diagnostic

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 4.8))

ax.axhline(0.0, linestyle='--', linewidth=1.2)

for Dtheta in Dtheta_values:
    tmp = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('v0').copy()
    if len(tmp) == 0:
        continue
    ax.plot(tmp['v0'], tmp['relative_asymmetry'],
            marker='o', linestyle='-', label=fr'$D_\theta={Dtheta:g}$')

ax.set_xscale('linear')
ax.set_xlabel(r'$v_0$')
ax.set_ylabel(r'$(\bar v_L - \bar v_R)/\bar v$')
ax.set_title(f'Left-right speed asymmetry, glued B=50R_inter, {main_method}')
ax.grid(False)
ax.legend(fontsize=12, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.tight_layout(rect=[0.0, 0.0, 0.80, 1.0])

save_path = figure_folder / 'glued_B50_active_v0_Dtheta_left_right_asymmetry.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 11. Optional active-effective FKPP diagnostic

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 5.2))

for Dtheta in Dtheta_values:
    tmp = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('D_eff').copy()
    if len(tmp) == 0:
        continue
    ax.plot(tmp['D_eff'], tmp['v2_measured'], marker='o', linestyle='', label=fr'$D_\theta={Dtheta:g}$')

valid = np.isfinite(summary['D_eff']) & np.isfinite(summary['v2_measured'])
if valid.sum() >= 2:
    D_line = np.linspace(summary.loc[valid, 'D_eff'].min(), summary.loc[valid, 'D_eff'].max(), 300)
    r_ref = float(summary.loc[valid, 'r_edge'].iloc[0])
    ax.plot(D_line, 4.0 * r_ref * D_line, '--', label=r'effective FKPP $4rD_{eff}$')

    slope, intercept = np.polyfit(summary.loc[valid, 'D_eff'], summary.loc[valid, 'v2_measured'], 1)
    ax.plot(D_line, slope * D_line + intercept, ':', label=fr'measured fit: slope={slope:.4g}')
    print(f'Measured pooled v^2 slope = {slope:.6g}')
    print(f'Effective FKPP slope      = {4.0 * r_ref:.6g}')
    print(f'Intercept                 = {intercept:.6g}')

ax.set_xscale('linear')
ax.set_yscale('linear')
ax.set_xlabel(r'$D_{eff}=D_r+v_0^2/(2D_\theta)$')
ax.set_ylabel(r'$v^2$')
ax.set_title('Squared speed versus active-effective diffusion, glued B=50R_inter')
ax.grid(False)
ax.legend(fontsize=11, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.tight_layout(rect=[0.0, 0.0, 0.76, 1.0])

save_path = figure_folder / 'glued_B50_active_v0_Dtheta_v2_vs_Deff.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 12. Threshold robustness

In [ ]:
def pooled_speed_summary_2d_by_method(fit_df, group_cols, methods, metadata_cols=None):
    rows = []
    for method in methods:
        tmp = run_level_speed_summary_2d(
            fit_df,
            group_cols=group_cols,
            method=method,
            metadata_cols=metadata_cols,
        )
        if len(tmp) > 0:
            rows.append(tmp)
    if len(rows) == 0:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)

threshold_df = pooled_speed_summary_2d_by_method(
    fit_df,
    group_cols=['Dtheta', 'v0'],
    methods=methods_to_check,
    metadata_cols=['Dr', 'p0', 'q0', 'r_edge', 'R_inter', 'dt', 'D_eff',
                   'persistence_time', 'persistence_length',
                   'isolation_buffer_factor', 'isolation_buffer', 'Lx', 'Ly'],
)

for Dtheta in Dtheta_values:
    tmpD = threshold_df[np.isclose(threshold_df['Dtheta'], Dtheta)].copy()
    if len(tmpD) == 0:
        continue

    fig, ax = plt.subplots(figsize=(7.8, 5.0))
    for method in methods_to_check:
        tmp = tmpD[tmpD['method'] == method].sort_values('v0')
        if len(tmp) == 0:
            continue
        ax.errorbar(tmp['v0'], tmp['mean_speed'], yerr=tmp['speed_std'],
                    marker='o', linestyle='-', capsize=3, label=method)

    ax.set_xscale('linear')
    ax.set_xlabel(r'$v_0$')
    ax.set_ylabel(r'$\langle v_{out}\rangle_{runs,sides}$')
    ax.set_title(fr'Threshold robustness, glued B=50R_inter, $D_\theta={Dtheta:g}$')
    ax.grid(False)
    ax.legend(fontsize=11)
    fig.tight_layout()

    save_path = figure_folder / f'glued_B50_threshold_methods_Dtheta_{safe_float_label(Dtheta)}.png'
    fig.savefig(save_path, dpi=170, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 13. Save fitted tables

In [ ]:
fit_path = figure_folder / 'glued_B50_active_v0_Dtheta_fit_table.csv'
summary_path = figure_folder / 'glued_B50_active_v0_Dtheta_summary.csv'
linear_fit_path = figure_folder / 'glued_B50_active_v0_Dtheta_linear_fits.csv'
threshold_path = figure_folder / 'glued_B50_active_v0_Dtheta_threshold_summary.csv'

fit_df.to_csv(fit_path, index=False)
summary.to_csv(summary_path, index=False)
linear_fit_df.to_csv(linear_fit_path, index=False)
threshold_df.to_csv(threshold_path, index=False)

print('Saved:', fit_path)
print('Saved:', summary_path)
print('Saved:', linear_fit_path)
print('Saved:', threshold_path)

## Fit-window sensitivity

In [ ]:
def window_label(a, b):
    return f'{a:.2f}-{b:.2f}'

window_rows = []
for f0, f1 in fit_windows:
    fit_df_w, missing_w = fit_front_sweep(
        run_table,
        data_folder=data_folder,
        param_base=param_base,
        methods=methods_to_check,
        fit_start_fraction=f0,
        fit_end_fraction=f1,
    )
    if len(fit_df_w) == 0:
        continue
    summary_w = run_level_speed_summary_2d(
        fit_df_w,
        group_cols=['Dtheta', 'v0'],
        method=main_method,
        metadata_cols=['r_edge', 'Dr', 'v_fkpp_brownian', 'v_fkpp_active_eff'],
    )
    summary_w['fit_window'] = window_label(f0, f1)
    window_rows.append(summary_w)

window_summary = pd.concat(window_rows, ignore_index=True) if window_rows else pd.DataFrame()
display(window_summary.head())

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary to plot yet.')
else:
    fig, axes = plt.subplots(len(fit_windows), len(Dtheta_values), figsize=(4.2 * len(Dtheta_values), 2.8 * len(fit_windows)), sharex=True, sharey=True)
    if len(fit_windows) == 1:
        axes = np.array([axes])
    for row_idx, (f0, f1) in enumerate(fit_windows):
        label = window_label(f0, f1)
        for col_idx, Dtheta in enumerate(Dtheta_values):
            ax = axes[row_idx, col_idx]
            sub = window_summary[(window_summary['fit_window'] == label) & np.isclose(window_summary['Dtheta'], Dtheta)].sort_values('v0')
            if len(sub) > 0:
                ax.errorbar(sub['v0'], sub['mean_speed'], yerr=sub['speed_sem'].fillna(0), marker='o', linestyle='-')
            if row_idx == 0:
                ax.set_title(fr'$D_\theta={Dtheta:g}$')
            if col_idx == 0:
                ax.set_ylabel(label + '\n' + r'$v_{front}$')
            if row_idx == len(fit_windows) - 1:
                ax.set_xlabel(r'$v_0$')
            style_axis(ax, n_ticks=4, grid=False)
    fig.tight_layout()
    fig.savefig(figure_folder / 'fit_window_sensitivity_active_v0_Dtheta.png', bbox_inches='tight')
    plt.show()

## Thesis plots

In [ ]:
# ==========================================================
# FINAL THESIS PLOTS: ACTIVE FRONT RESULTS
# ==========================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter

# ----------------------------------------------------------
# Output folder
# ----------------------------------------------------------

thesis_figure_folder = figure_folder / "thesis_final_active_front"
thesis_figure_folder.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------
# Plot style
# ----------------------------------------------------------

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 15,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "lines.markersize": 6,
})

def style_axis(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.tick_params(direction="out", length=4, width=1)
    return ax

def save_thesis_figure(fig, name):
    pdf_path = thesis_figure_folder / f"{name}.pdf"
    png_path = thesis_figure_folder / f"{name}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print("Saved:", pdf_path)
    print("Saved:", png_path)

# ----------------------------------------------------------
# Prepare data
# ----------------------------------------------------------

df = summary.copy()

error_column = "speed_sem"

# Preserve your original Dtheta ordering if it exists
try:
    Dtheta_plot_values = list(Dtheta_values)
except NameError:
    Dtheta_plot_values = sorted(df["Dtheta"].unique())

Dtheta_plot_values = [float(D) for D in Dtheta_plot_values]

r_ref = float(df["r_edge"].iloc[0])
Dr_ref = float(df["Dr"].iloc[0])
v_fkpp_brownian = 2.0 * np.sqrt(Dr_ref * r_ref)

# Reference speed at v0 = 0 for each Dtheta
baseline = (
    df[np.isclose(df["v0"], 0.0)]
    [["Dtheta", "mean_speed", error_column]]
    .rename(columns={
        "mean_speed": "v_ref",
        error_column: "v_ref_err",
    })
)

df = df.merge(baseline, on="Dtheta", how="left")

df["x_active"] = df["v0"]**2 / df["Dtheta"]
df["y_active_measured"] = df["mean_speed"]**2 - df["v_ref"]**2

df["yerr_active_measured"] = np.sqrt(
    (2.0 * df["mean_speed"] * df[error_column])**2
    +
    (2.0 * df["v_ref"] * df["v_ref_err"])**2
)

display(df.head())

In [ ]:
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))


color_map = {
    0.002: "tab:blue",
    0.05:  "tab:orange",
    0.5:   "tab:green",
    2.0:   "tab:purple",
    5.0:   "tab:olive",
}

legend_handles = []
legend_labels = []


ax = axes[0]

for Dtheta in Dtheta_plot_values:
    sub = df[np.isclose(df["Dtheta"], Dtheta)].sort_values("v0")
    color = color_map[float(Dtheta)]

    cont = ax.errorbar(
        sub["v0"],
        sub["mean_speed"],
        yerr=sub[error_column],
        marker="o",
        linestyle="",
        capsize=3,
        color=color,
        label=fr"$D_\theta={Dtheta:g}$",
    )

    legend_handles.append(cont)
    legend_labels.append(fr"$D_\theta={Dtheta:g}$")

    fit_sub = sub[sub["v0"] > 0].copy()
    if len(fit_sub) >= 2:
        slope, intercept = np.polyfit(fit_sub["v0"], fit_sub["mean_speed"], 1)
        x_line = np.linspace(fit_sub["v0"].min(), fit_sub["v0"].max(), 200)
        ax.plot(
            x_line,
            slope * x_line + intercept,
            linestyle="--",
            linewidth=1.6,
            color=color,
        )

ax.set_xlabel(r"$v_0$")
ax.set_ylabel(r"$v_{\rm front}$")
ax.set_title(r"(a) Front speed")
style_axis(ax, nbins=4)


ax = axes[1]

for Dtheta in Dtheta_plot_values:
    sub = df[np.isclose(df["Dtheta"], Dtheta)].sort_values("v0")
    color = color_map[float(Dtheta)]

    ax.errorbar(
        sub["x_active"],
        sub["y_active_measured"],
        yerr=sub["yerr_active_measured"],
        marker="o",
        markersize = 7,
        linestyle="",
        capsize=3,
        color=color,
    )

ax.axhline(0.0, linestyle=":", linewidth=1.1, color="0.5")

x_line_reference = np.linspace(0.0, 0.00035, 300)
ax.plot(
    x_line_reference,
    2.0 * r_ref * x_line_reference,
    linestyle="--",
    linewidth=1.8,
    color="red",
)

ax.set_xlim(-0.000005, 0.00015)
ax.set_ylim(-0.00001, 0.0002)

ax.set_xlabel(r"$v_0^2/D_\theta$")
ax.set_ylabel(r"$\Delta v_{\rm act}^2$")
ax.set_title(r"(b) Activity-induced excess")
style_axis(ax, nbins=4)


reference_slope_label = fr"reference slope: $2r={2.0*r_ref:.3g}$"

reference_slope_handle = Line2D(
    [0], [0],
    color="red",
    linestyle="--",
    linewidth=1.8,
    label=reference_slope_label,
)

fig.legend(
    legend_handles + [reference_slope_handle],
    legend_labels + [reference_slope_label],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.04),
    ncol=6,
    frameon=False,
    handlelength=1.8,
    columnspacing=1.6,
    fontsize = 16,
)

fig.tight_layout(rect=[0.0, 0.10, 1.0, 1.0])

save_thesis_figure(fig, "active_front_velocity_and_measured_excess")
plt.show()

In [ ]:
fit_results_measured_ref = []

color_map = {
    0.002: "tab:blue",
    0.05:  "tab:orange",
    0.5:   "tab:green",
    2.0:   "tab:purple",
    5.0:   "tab:olive",
}

fig = plt.figure(figsize=(13.5, 8.2))
gs = fig.add_gridspec(2, 6, hspace=0.55, wspace=0.75)

axes = [
    fig.add_subplot(gs[0, 0:2]),
    fig.add_subplot(gs[0, 2:4]),
    fig.add_subplot(gs[0, 4:6]),
    fig.add_subplot(gs[1, 1:3]),
    fig.add_subplot(gs[1, 3:5]),
]

for ax, Dtheta in zip(axes, Dtheta_plot_values):
    sub = df[np.isclose(df["Dtheta"], Dtheta)].sort_values("v0").copy()
    color = color_map[float(Dtheta)]

    x = sub["x_active"].to_numpy()
    y = sub["y_active_measured"].to_numpy()
    yerr = sub["yerr_active_measured"].to_numpy()

    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(yerr)
    x = x[mask]
    y = y[mask]
    yerr = yerr[mask]

    slope, intercept = np.polyfit(x, y, 1)

    y_pred = slope * x + intercept
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    fit_results_measured_ref.append({
        "Dtheta": Dtheta,
        "slope": slope,
        "intercept": intercept,
        "R2": r2,
        "n_points": len(x),
    })

    ax.errorbar(
        x,
        y,
        yerr=yerr,
        marker="o",
        linestyle="",
        capsize=3,
        color=color,
    )

    x_line = np.linspace(np.nanmin(x), np.nanmax(x), 300)
    ax.plot(
        x_line,
        slope * x_line + intercept,
        linestyle="--",
        linewidth=1.8,
        color=color,
    )

    ax.axhline(0.0, linestyle=":", linewidth=1.0, color="0.55")

    ax.text(
        0.05,
        0.92,
        fr"$D_\theta={Dtheta:g}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )
    ax.text(
        0.05,
        0.80,
        fr"slope $\simeq {slope:.3g}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_xlabel(r"$v_0^2/D_\theta$")
    ax.set_ylabel(r"$\Delta v_{\rm act}^2$")
    style_axis(ax, nbins=4)

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-2, 2))
    ax.yaxis.set_major_formatter(formatter)
    ax.xaxis.set_major_formatter(formatter)

save_thesis_figure(fig, "active_front_measured_excess_separate_Dtheta_fits")
plt.show()

fit_results_measured_ref_df = pd.DataFrame(fit_results_measured_ref)
display(fit_results_measured_ref_df)